In [10]:
import os
import zipfile
import requests
import pandas as pd
from pathlib import Path

# Konfiguracja stacji
STATION_ID = "566"  # Kraków-Balice
STATION_NAME = "KRAKOW_BALICE"

class IMGWDataFetcher:
    BASE_URL = "https://danepubliczne.imgw.pl/data/dane_pomiarowo_obserwacyjne/dane_meteorologiczne/terminowe/synop/"

    def __init__(self, download_dir: Path = Path("dataset/imgw/balice")):
        self.download_dir = download_dir
        self.download_dir.mkdir(parents=True, exist_ok=True)

    def get_year_folder(self, year: int) -> str:
        """Określa folder na serwerze IMGW."""
        if 1996 <= year <= 2000:
            return "1996_2000"
        return str(year)

    def fetch_balice_data(self, start_year: int, end_year: int):
        """Pobiera i przetwarza dane rok po roku."""
        for year in range(start_year, end_year + 1):
            year_folder = self.get_year_folder(year)
            
            # Obsługa różnych nazw plików ZIP (wyjątek roku 2000)
            if year == 2000:
                zip_filename = f"1996_2000_{STATION_ID}_s.zip"
            else:
                zip_filename = f"{year}_{STATION_ID}_s.zip"
            
            url = f"{self.BASE_URL}{year_folder}/{zip_filename}"
            print(f"--- Przetwarzanie roku {year} ---")
            
            try:
                response = requests.get(url, timeout=20)
                if response.status_code == 200:
                    zip_path = self.download_dir / f"temp_{zip_filename}"
                    with open(zip_path, "wb") as f:
                        f.write(response.content)
                    
                    self._extract_temperature(zip_path, year)
                    zip_path.unlink()  # Usuwamy tymczasowy ZIP
                else:
                    print(f"   ⚠️ Brak pliku na serwerze: {url} (Status: {response.status_code})")
            except Exception as e:
                print(f"   ❌ Błąd połączenia przy roku {year}: {e}")

    def _extract_temperature(self, zip_path: Path, year: int):
        """Wypakowuje CSV, wybiera kolumny czasu i temperatury."""
        with zipfile.ZipFile(zip_path, "r") as z:
            all_files = z.namelist()
            # Szukamy pliku CSV zawierającego numer stacji 566
            csv_candidates = [f for f in all_files if STATION_ID in f and f.endswith('.csv')]
            
            if csv_candidates:
                target_csv = csv_candidates[0]
                with z.open(target_csv) as f:
                    # Wczytujemy dane (latin2 dla polskich znaków)
                    df = pd.read_csv(f, header=None, encoding='iso-8859-2', low_memory=False)
                    
                    # Mapowanie kolumn zgodnie z Twoją listą:
                    # Indeks 2: ROK, 3: MC, 4: DZ, 5: GG, 29: TEMP
                    try:
                        df_out = df[[2, 3, 4, 5, 29]].copy()
                        df_out.columns = ['year', 'month', 'day', 'hour', 'temp']
                        
                        # Czyszczenie danych: zamiana tekstu na liczby, błędy na NaN
                        df_out['temp'] = pd.to_numeric(df_out['temp'], errors='coerce')
                        
                        # Filtrujemy tylko rok, którego dotyczy pętla
                        # (Ważne dla paczki 1996_2000, która ma 5 lat w jednym pliku)
                        df_filtered = df_out[df_out['year'] == year].copy()
                        
                        if not df_filtered.empty:
                            output_file = self.download_dir / f"{STATION_NAME}_{year}.csv"
                            df_filtered.to_csv(output_file, index=False)
                            print(f"   ✅ Zapisano: {output_file.name} ({len(df_filtered)} wierszy)")
                        else:
                            print(f"   ⚠️ Brak rekordów dla roku {year} wewnątrz pliku CSV.")
                            
                    except IndexError:
                        print(f"   ❌ Błąd struktury pliku CSV w roku {year}. Sprawdź liczbę kolumn.")
            else:
                print(f"   ❓ Nie znaleziono pliku CSV wewnątrz ZIP dla roku {year}")

# --- URUCHOMIENIE ---
if __name__ == "__main__":
    fetcher = IMGWDataFetcher()
    # Zakres lat zgodny z Twoim ERA5-Land
    fetcher.fetch_balice_data(2000, 2025)

--- Przetwarzanie roku 2000 ---
   ✅ Zapisano: KRAKOW_BALICE_2000.csv (8784 wierszy)
--- Przetwarzanie roku 2001 ---
   ✅ Zapisano: KRAKOW_BALICE_2001.csv (8760 wierszy)
--- Przetwarzanie roku 2002 ---
   ✅ Zapisano: KRAKOW_BALICE_2002.csv (8760 wierszy)
--- Przetwarzanie roku 2003 ---
   ✅ Zapisano: KRAKOW_BALICE_2003.csv (8760 wierszy)
--- Przetwarzanie roku 2004 ---
   ✅ Zapisano: KRAKOW_BALICE_2004.csv (8784 wierszy)
--- Przetwarzanie roku 2005 ---
   ✅ Zapisano: KRAKOW_BALICE_2005.csv (8760 wierszy)
--- Przetwarzanie roku 2006 ---
   ✅ Zapisano: KRAKOW_BALICE_2006.csv (8760 wierszy)
--- Przetwarzanie roku 2007 ---
   ✅ Zapisano: KRAKOW_BALICE_2007.csv (8760 wierszy)
--- Przetwarzanie roku 2008 ---
   ✅ Zapisano: KRAKOW_BALICE_2008.csv (8784 wierszy)
--- Przetwarzanie roku 2009 ---
   ✅ Zapisano: KRAKOW_BALICE_2009.csv (8760 wierszy)
--- Przetwarzanie roku 2010 ---
   ✅ Zapisano: KRAKOW_BALICE_2010.csv (8760 wierszy)
--- Przetwarzanie roku 2011 ---
   ✅ Zapisano: KRAKOW_BALICE_2011

In [ ]:
import os
import zipfile
import requests
import pandas as pd
import io
from pathlib import Path

# Konfiguracja
STATION_ID = 250190390  # Kraków-Obserwatorium
STATION_NAME = "KRAKOW_OBSERWATORIUM"

class IMGWKlimatFetcher:
    BASE_URL = "https://danepubliczne.imgw.pl/data/dane_pomiarowo_obserwacyjne/dane_meteorologiczne/terminowe/klimat/"

    def __init__(self, download_dir: Path = Path("dataset/imgw/obserwatorium")):
        self.download_dir = download_dir
        self.download_dir.mkdir(parents=True, exist_ok=True)

    def fetch_data(self, start_year: int, end_year: int):
        all_results = []

        for year in range(start_year, end_year + 1):
            print(f"\n--- Przetwarzanie roku {year} ---")
            
            # Próba 1: Czy dane są miesięczne (standard po 2000)
            found_any_month = False
            for month in range(1, 13):
                # Sprawdzamy oba możliwe foldery dla roku 2000
                folders_to_check = [str(year)]
                if year == 2000:
                    folders_to_check.append("1996_2000")

                for folder in folders_to_check:
                    zip_filename = f"{year}_{month:02d}_k.zip"
                    url = f"{self.BASE_URL}{folder}/{zip_filename}"
                    
                    df_month = self._download_and_process(url, year, month)
                    if df_month is not None:
                        all_results.append(df_month)
                        found_any_month = True
                        break # Znaleźliśmy w jednym z folderów, nie szukaj dalej

            # Próba 2: Jeśli nie znaleziono plików miesięcznych, szukamy rocznej paczki (częste w 2000)
            if not found_any_month:
                print(f"   ℹ️ Brak plików miesięcznych dla {year}, szukam paczki rocznej...")
                for folder in [str(year), "1996_2000"]:
                    zip_filename = f"{year}_k.zip" # Twoja poprawka!
                    url = f"{self.BASE_URL}{folder}/{zip_filename}"
                    df_year = self._download_and_process(url, year)
                    if df_year is not None:
                        all_results.append(df_year)
                        break

        if all_results:
            final_df = pd.concat(all_results, ignore_index=True)
            final_df['temp'] = pd.to_numeric(final_df['temp'], errors='coerce')
            output_file = self.download_dir / "krakow_obserwatorium_imgw_2000_2025.csv"
            final_df.to_csv(output_file, index=False)
            print(f"\n✅ Zakończono! Łącznie pobrano {len(final_df)} rekordów.")
        else:
            print("\n❌ Nie znaleziono żadnych danych.")

    def _download_and_process(self, url, year, month=None):
        """Pobiera ZIP i wyciąga dane dla stacji."""
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                    # Szukamy pliku CSV (może być k_t_mm_rrrr.csv lub k_t_rrrr.csv)
                    csv_pattern = f"k_t_{month:02d}_{year}.csv" if month else f"k_t_{year}.csv"
                    
                    # Czasami nazwa w środku jest inna, np. bez miesiąca
                    all_csvs = [f for f in z.namelist() if f.endswith('.csv')]
                    target_csv = csv_pattern if csv_pattern in all_csvs else (all_csvs[0] if all_csvs else None)

                    if target_csv:
                        with z.open(target_csv) as f:
                            df = pd.read_csv(f, header=None, encoding='iso-8859-2', low_memory=False)
                            df_krakow = df[df[0] == STATION_ID].copy()
                            if not df_krakow.empty:
                                # Wybieramy: Rok(2), Miesiąc(3), Dzień(4), Godzina(5), Temp(6)
                                res = df_krakow[[2, 3, 4, 5, 6]].copy()
                                res.columns = ['year', 'month', 'day', 'hour', 'temp']
                                # Na wszelki wypadek filtrujemy po roku (paczki wieloletnie)
                                res = res[res['year'] == year]
                                if month: print(f"   ✅ Pobrano: {month:02d}/{year}")
                                else: print(f"   ✅ Pobrano paczkę roczną {year}")
                                return res
            return None
        except:
            return None

if __name__ == "__main__":
    fetcher = IMGWKlimatFetcher()
    fetcher.fetch_data(2000, 2025)


--- Przetwarzanie roku 2000 ---
   ℹ️ Brak plików miesięcznych dla 2000, szukam paczki rocznej...
   ✅ Pobrano paczkę roczną 2000

--- Przetwarzanie roku 2001 ---
   ✅ Pobrano: 01/2001 (Stacji: 2)
   ✅ Pobrano: 02/2001 (Stacji: 2)
   ✅ Pobrano: 03/2001 (Stacji: 2)
   ✅ Pobrano: 04/2001 (Stacji: 2)
   ✅ Pobrano: 05/2001 (Stacji: 2)
   ✅ Pobrano: 06/2001 (Stacji: 2)
   ✅ Pobrano: 07/2001 (Stacji: 2)
   ✅ Pobrano: 08/2001 (Stacji: 2)
   ✅ Pobrano: 09/2001 (Stacji: 2)
   ✅ Pobrano: 10/2001 (Stacji: 2)
   ✅ Pobrano: 11/2001 (Stacji: 2)
   ✅ Pobrano: 12/2001 (Stacji: 2)

--- Przetwarzanie roku 2002 ---
   ✅ Pobrano: 01/2002 (Stacji: 2)
   ✅ Pobrano: 02/2002 (Stacji: 2)
   ✅ Pobrano: 03/2002 (Stacji: 2)
   ✅ Pobrano: 04/2002 (Stacji: 2)
   ✅ Pobrano: 05/2002 (Stacji: 2)
   ✅ Pobrano: 06/2002 (Stacji: 2)
   ✅ Pobrano: 07/2002 (Stacji: 2)
   ✅ Pobrano: 08/2002 (Stacji: 2)
   ✅ Pobrano: 09/2002 (Stacji: 2)
   ✅ Pobrano: 10/2002 (Stacji: 2)
   ✅ Pobrano: 11/2002 (Stacji: 2)
   ✅ Pobrano: 12/2002